# Control-reference residual CNN extensions

**New extensions only: this notebook does not train original LPGNet or a raw-signal baseline.** A separate notebook already covers that reference. Here, compare:
1. **LPR-11 + statistics pooling:** order 11, sensor-wise fusion, tiny separable CNN, temporal mean/std.
2. **Multiscale LPR + statistics pooling:** orders 5/11/21, the same fusion output width, CNN and mean/std head.

Reference motivation: [Alle & Priyakumar, 2021](https://arxiv.org/abs/2107.12878), [original implementation](https://github.com/devalab/parkinsonsfromgait). This is an independent PyTorch/SciPy extension, not numerical reproduction of their MATLAB/TensorFlow pipeline.

**10 matched outer participant folds, 3 training seeds, inner-only selection.** Predictor coefficients and channel scales are fitted on inner-training controls only. Exact cross-identity duplicates in the official dataset are quarantined before splitting. All clinical analyses are post-evaluation.

No Mamba installation is required. GPU is recommended; real CPU execution is also supported. Outputs remain in RAM unless explicitly exported. Prior results on this cohort motivated the design: the new run is a follow-up, not external confirmation.

In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, json, copy, zipfile

SOURCE_REV = '68b3f2ade464780625fcbee7086c53895c5221d9'
REPO_URL = 'https://github.com/AlexM3020/Subject-Independent-Evaluation-of-Mamba-for-Parkinson-s-Disease-Detection-from-Gait-Signals.git'
INSTALL_DEPENDENCIES = False  # Only if base PyTorch/SciPy/sklearn/Matplotlib are missing.
roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for b in roots for p in (b,b/'paper_repository') if (p/'gait/residual_experiment.py').is_file()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path(tempfile.gettempdir()) / ('lpr-extensions-' + SOURCE_REV[:12])
    if not REPO_ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_ROOT)],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',SOURCE_REV],check=True)
REPO_ROOT = REPO_ROOT.resolve()
SOURCE_FILES = ('__init__.py','models.py','cross_validation.py','official_data.py','diagnostics.py','residual_experiment.py')
for name in SOURCE_FILES:
    expected = subprocess.run(['git','-C',str(REPO_ROOT),'show',SOURCE_REV+':gait/'+name],capture_output=True,check=True).stdout
    actual = (REPO_ROOT/'gait'/name).read_bytes()
    assert actual.replace(b'\r\n',b'\n') == expected.replace(b'\r\n',b'\n'), f'Changed source: {name}'
# Detect the stale-package problem before importing the new engine. Do not clear live results.
stale = [name for name,module in list(sys.modules.items()) if (name=='gait' or name.startswith('gait.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(REPO_ROOT)]
if stale:
    raise RuntimeError('An older gait package is cached. Export needed results, restart the kernel, then run THIS setup first. Stale modules: '+str(stale))
sys.path.insert(0,str(REPO_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_ROOT/'requirements-lpr.txt')],check=True)
print('Verified source:',SOURCE_REV)
print('Repository:',REPO_ROOT)

## Fixed configuration and budget
50-Hz residuals use a causal two-point moving average and decimation. Both arms discard the first 22 downsampled samples; order 11 is not given extra boundary samples. The normal-reference estimator is participant/recording-balanced ridge Yule–Walker, not a per-test-recording fit. Residual energy is preserved: no per-window residual z-score.

Both models use a grouped 1×1 fusion (two outputs per sensor), separable conv widths 32/32/64, kernels 7/5/3, channel LayerNorm + ELU, max pooling after the first two blocks, and temporal mean/std. Neural counts: 5,446 versus 5,518; predictor coefficients are additional (198 versus 666 used non-leading coefficients).

Default **2 arms × 10 folds × 3 seeds × 3 candidates = 180 two-stage fits**. One fixed candidate plus two deterministic random optimizer/dropout candidates. Stage 1: up to 60 epochs of 2-second, nonoverlapping windows, weighted window CE. Stage 2: up to 40 epochs of frozen-encoder head calibration, unweighted participant CE. Both select using inner participant NLL. Epoch-zero head is retained if calibration does not help.

Each invocation defaults to two complete tasks (both arms for one fold/seed). Resume with another invocation, or explicitly export and reload in a later session. Set `MAX_NEW_TASKS=None` to finish all remaining tasks. Do not inspect outer scores or change the frozen plan partway through.

In [ ]:
from dataclasses import asdict
from collections import Counter
from html import escape
from IPython.display import display, HTML
import numpy as np
import torch
import matplotlib.pyplot as plt
from gait import residual_experiment as lpr
from gait import official_data as official
from gait.cross_validation import export_results

CACHE_DIR = Path(tempfile.gettempdir())/'gaitpdb-1.0.0-verified-cache'
ALLOW_DOWNLOAD = True  # Explicit verified input cache; attached Kaggle files stay untouched.
COMMON_MIN_SAMPLES = 2000
SEARCH_SEED = 9011
CANDIDATE_COUNT = 3
CFG = lpr.ResidualProtocol(outer_folds=10,inner_fraction=.2,split_seed=42,
    training_seeds=(42,123,2026),window_samples=100,batch_size=64,
    encoder_epochs=60,head_epochs=40,patience=10,scheduler_patience=4,
    bootstrap_samples=2000,bootstrap_seed=7301,head_lr=1e-3,ridge=1e-4)
CANDIDATES = lpr.candidate_search(CANDIDATE_COUNT,SEARCH_SEED)
DEVICE = 'cuda'  # Set 'cpu' explicitly for slower real CPU training.
RUN_EXPERIMENT = False
MAX_NEW_TASKS = 2
lpr_results = globals().get('lpr_results',{})
lpr.validate(CFG,CANDIDATES)
PLAN = json.loads(json.dumps({'protocol':asdict(CFG),'candidates':CANDIDATES,'source_revision':SOURCE_REV,
                              'search_seed':SEARCH_SEED,'common_min_samples':COMMON_MIN_SAMPLES,'device':DEVICE}))
print(json.dumps(PLAN,indent=2))
print('Two-stage fits:',2*CFG.outer_folds*len(CFG.training_seeds)*len(CANDIDATES))
print(lpr.runtime())

def show_table(rows,title='',columns=None,limit=None):
    rows=list(rows)
    print(title,'| rows:',len(rows))
    if not rows: return
    columns=columns or list(dict.fromkeys(k for r in rows for k in r))
    def fmt(value):
        if value is None: return 'Unknown / not applicable'
        return f'{value:.4f}' if isinstance(value,float) else str(value)
    shown=rows if limit is None else rows[:limit]
    html='<div style="max-height:480px;overflow:auto"><table><thead><tr>'+''.join('<th>'+escape(c)+'</th>' for c in columns)+'</tr></thead><tbody>'
    html+=''.join('<tr>'+''.join('<td>'+escape(fmt(r.get(c)))+'</td>' for c in columns)+'</tr>' for r in shown)
    display(HTML(html+'</tbody></table></div>'))
    if limit is not None and len(rows)>limit: print('Full table stays in RAM and optional export.')

## Verified common cohort and clinical metadata
All 306 official original recordings and metadata formats are checksum-verified. Exclude all recordings of GaPt17, GaPt18, JuCo26 and JuPt01 due to cross-identity duplicate signals (including a conflicting diagnosis pair). Within-identity exact copies count once. The expected curated cohort is 161 participants and 292 recordings. No model error informs exclusion.

HY joins use exact study-qualified IDs and agreement between official TXT and HTML. Missing/ambiguous entries stay Unknown. Metadata is not passed to the training API. Whole-cohort hashes serve only integrity/curation; reference fitting never uses validation/test controls.

In [ ]:
minimum=max(COMMON_MIN_SAMPLES,2*(CFG.window_samples+lpr.WARMUP))
records,audit,provenance=official.load_verified_cohort(CACHE_DIR,minimum,ALLOW_DOWNLOAD)
clinical,clinical_provenance=official.load_verified_clinical(CACHE_DIR,ALLOW_DOWNLOAD)
splits=lpr.cv.make_splits(records,CFG)
labels={r['subject']:r['label'] for r in records}
print('Curation:',provenance)
print('File audit:',dict(Counter(r['status'] for r in audit)))
print('Participants:',dict(Counter(labels.values())),'| recordings:',len(records))
print('Clinical issues:',clinical_provenance['issues'])
show_table([{'fold':s['fold'],**{p:dict(Counter(labels[x] for x in s[p])) for p in ('train','validation','test')}} for s in splits],title='Shared participant partitions (0=Control,1=PD)')
print('No LP coefficients or normalization statistics have been fitted on the full cohort.')
FROZEN_PLAN=copy.deepcopy(PLAN)

## Real network check (no research training)
Forward/backward checks cover both channel-bank sizes and multiple input lengths. No CUDA extensions or substitute model are used. Run on the same chosen device as training. A smoke check is not an accuracy result or an end-to-end inference benchmark.

In [ ]:
MODEL_READY=False
if DEVICE=='cuda' and not torch.cuda.is_available():
    print('Enable a Kaggle GPU, or explicitly select CPU before freezing the plan.')
else:
    smoke=[]
    for name,orders in lpr.VARIANTS.items():
        model=lpr.ResidualStatsCNN(len(orders),.25).to(DEVICE)
        for length in (CFG.window_samples,1000):
            model.zero_grad(set_to_none=True)
            out=model(torch.randn(2,length,18*len(orders),device=DEVICE))
            assert out.shape==(2,2) and torch.isfinite(out).all()
            torch.nn.functional.cross_entropy(out,torch.tensor([0,1],device=DEVICE)).backward()
            assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        smoke.append({'variant':name,'orders':orders,'neural_parameters':sum(p.numel() for p in model.parameters()),
                      'used_lp_coefficients':18*sum(orders),'residual_receptive_field_samples':model.receptive_field_samples})
        del model,out
    if DEVICE=='cuda': torch.cuda.empty_cache()
    show_table(smoke,title='Actual forward/backward checks passed')
    MODEL_READY=True
    MODEL_PLAN=copy.deepcopy(PLAN)

## Optional completed-task resume
Upload an explicitly exported LPR progress ZIP to resume. It contains JSON, not executable/pickled weights. Source, full plan, cohort, runtime, participant predictions and control references are checked when resuming. Interrupted tasks restart both stages. Do not load a previous Mamba/CNN comparison bundle into this experiment.

In [ ]:
RESUME_ZIP=''
if RESUME_ZIP:
    if lpr_results.get('runs'): raise RuntimeError('Existing results in RAM; export before clearing them explicitly')
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        lpr_results=json.loads(archive.read('results.json'))
    print('Loaded tasks:',len(lpr_results.get('runs',[])),'; full validation occurs at resume')
else:
    print('No progress loaded.')

## Train / continue
Reference coefficients and scale are fitted only on each fold's inner-training controls. Stage 1 trains weighted window CE, selecting against full-recording-derived inner participant NLL. Stage 2 freezes the encoder and calibrates the head on one embedding per inner-training participant. Its unweighted objective and fixed LR 1e-3 differ intentionally from stage 1; candidate weight decay is shared between stages. Both stages use inner validation only.

Final scoring: full unpadded recording → mean/std feature vector → equal mean over recordings of that participant → selected head. This is **feature aggregation**, not averaging window probabilities. No outer window accuracy is reported. No refit on outer development, no threshold tuning; PD probability threshold is 0.5. Both new variants are reported; the original LPGNet is not run.

No test scores are printed until every planned task is complete. Change `MAX_NEW_TASKS` freely to manage session time; do not change scientific configuration after freezing it.

In [ ]:
if RUN_EXPERIMENT:
    current=json.loads(json.dumps({'protocol':asdict(CFG),'candidates':CANDIDATES,'source_revision':SOURCE_REV,
        'search_seed':SEARCH_SEED,'common_min_samples':COMMON_MIN_SAMPLES,'device':DEVICE}))
    assert current==FROZEN_PLAN,'Plan changed after audit; start a new declared experiment'
    assert MODEL_READY and MODEL_PLAN==current,'Run the actual model check for this plan first'
    lpr.run(records,audit,provenance,CFG,CANDIDATES,progress=lpr_results,
            max_new_tasks=MAX_NEW_TASKS,verbose=False,device=DEVICE)
    lpr_results['source_revision']=SOURCE_REV
    lpr_results['search_seed']=SEARCH_SEED
    lpr_results['clinical_provenance']=clinical_provenance
    print('Completed tasks:',len(lpr_results['runs']),'/',2*CFG.outer_folds*len(CFG.training_seeds))
    print('All planned tasks complete:',lpr_results['complete'])
else:
    print('Training disabled. Set RUN_EXPERIMENT=True after the audit and model check.')

## Explicit optional export
Export completed tasks before a Kaggle session reset if you want to continue later. Nothing is automatically saved. The ZIP includes the complete plan, file audit, participant splits, fitted LP filters/scales, inner histories/predictions, selected settings and outer participant predictions. No raw signals or weights. Partial bundles are resumable only at completed-task boundaries, not mid-epoch. Rerun after diagnostics to include tables.

In [ ]:
SAVE_RESULTS=False
if SAVE_RESULTS:
    from datetime import datetime, timezone
    if not lpr_results.get('runs'): raise RuntimeError('No completed task to export')
    lpr_results['source_revision']=SOURCE_REV
    lpr_results['search_seed']=SEARCH_SEED
    lpr_results['clinical_provenance']=clinical_provenance
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    path=Path('/kaggle/working')/f'lpr_extensions_{stamp}.zip'
    print('Saved:',export_results(lpr_results,path))
else:
    print('No result files written. Results remain in RAM.')

## Participant OOF results and paired uncertainty
Each person contributes exactly one outer prediction per variant/seed. Bootstrap intervals resample participants within diagnosis, conditional on fitted OOF predictions and class counts. They do not capture full refitting or new-cohort uncertainty. Fold SD is descriptive, not a CI. Paired differences resample the same people across both variants and all seeds, averaging seed-specific metric differences—not ensemble predictions.

Clinical/study and high-loss diagnostics are exploratory. An elevated LPR response is departure from a control reference, not proof of a PD-specific mechanism. Unknown clinical values remain explicit; inner-validation rows may repeat people across folds and are not independent test observations.

In [ ]:
if not lpr_results.get('complete'):
    print('Finish all planned tasks before examining outer results.')
else:
    print('Building paired participant diagnostics; bootstrap may take some time.')
    lpr_diagnostics=lpr.reports(lpr_results,clinical)
    lpr_results['diagnostics']=lpr_diagnostics
    metrics=[]
    for family,summaries in lpr_results['summaries'].items():
        for seed,summary in summaries.items():
            for name,interval in summary['confidence_intervals'].items():
                metrics.append({'variant':family,'seed':seed,'metric':name,**interval})
    show_table(metrics,title='Participant OOF metrics and 95% intervals')
    show_table([{'variant':f,'seed':s,**v['participant_metrics'],'fold_accuracy_sd':v['fold_accuracy_std']} for f,ss in lpr_results['summaries'].items() for s,v in ss.items()],title='Counts, NLL and descriptive fold SD')
    across=[]
    for family,summaries in lpr_results['summaries'].items():
        for metric in ('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc'):
            values=[s['participant_metrics'][metric] for s in summaries.values()]
            across.append({'variant':family,'metric':metric,'seed_mean':float(np.mean(values)),
                           'seed_sample_sd':float(np.std(values,ddof=1)) if len(values)>1 else None})
    show_table(across,title='All training seeds, without selecting the best')
    show_table(lpr_diagnostics['paired_comparison'],title='Multiscale minus single-order: paired participant bootstrap')

In [ ]:
if lpr_results.get('complete') and 'diagnostics' in lpr_results:
    report=lpr_results['diagnostics']
    show_table(report['participant_errors'],title='Complete participant error table',columns=['family','seed','fold','subject','study','diagnosis','probability','prediction','correct','recordings','hy','hy_status','confidence','nll'])
    for key in ('by_study','by_study_diagnosis','by_fold','pd_severity','metadata_coverage','confidently_wrong'):
        show_table(report[key],title=key)
    show_table(report['inner_high_loss'],title='Selected inner-checkpoint NLL contributors (not independent across folds)',limit=30)
    show_table(sorted(report['seed_stability'],key=lambda r:(-r['error_seeds'],r['family'],r['subject'])),title='Persistent errors and training-seed variability')
    trials=[]
    for r in lpr_results['runs']:
        for index,t in enumerate(r['trials']):
            trials.append({'variant':r['family'],'seed':r['seed'],'fold':r['fold'],**t['candidate'],
                'selected':index==r['selected_candidate'],'orders':t['orders'],'parameters':t['parameters'],
                'best_encoder_epoch':t['best_encoder_epoch'],'best_head_epoch':t['best_head_epoch'],
                'inner_nll':t['inner_metrics']['log_loss'],'seconds':t['seconds'],'peak_allocated_bytes':t['peak_allocated_bytes']})
    lpr_results['trial_table']=trials
    show_table(trials,title='All inner search trials',limit=40)
    print('Candidate training hours:',sum(t['seconds'] for t in trials)/3600)
else:
    print('Run completed-results analysis first.')

In [ ]:
# Display only; no automatic files, smoothing or repeated-value history padding.
if lpr_results.get('complete'):
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
    for seed in lpr_results['protocol']['training_seeds']:
        fig,axes=plt.subplots(2,2,figsize=(11,9))
        for i,family in enumerate(lpr_results['variants']):
            rows=[p for r in lpr_results['runs'] if r['family']==family and r['seed']==seed for p in r['test_participants']]
            y=np.array([r['label'] for r in rows]); p=np.array([r['probability'] for r in rows])
            ConfusionMatrixDisplay(confusion_matrix(y,p>=.5,labels=[0,1]),display_labels=['Control','PD']).plot(ax=axes[i,0],colorbar=False)
            axes[i,0].set_title(f'{family} | seed {seed}')
            fpr,tpr,_=roc_curve(y,p)
            axes[i,1].plot(fpr,tpr,label=f'AUC={roc_auc_score(y,p):.3f}')
            axes[i,1].plot([0,1],[0,1],'--',color='gray'); axes[i,1].legend()
            axes[i,1].set(xlabel='False positive rate',ylabel='True positive rate',title='Participant OOF ROC')
        fig.tight_layout(); plt.show()
        fig,axes=plt.subplots(2,2,figsize=(12,8))
        for i,family in enumerate(lpr_results['variants']):
            for r in lpr_results['runs']:
                if r['family']!=family or r['seed']!=seed: continue
                trial=r['trials'][r['selected_candidate']]
                for j,key in enumerate(('history','head_history')):
                    history=trial[key]
                    axes[i,j].plot([h['epoch'] for h in history],[h['validation_log_loss'] for h in history],label=f"Fold {r['fold']}",alpha=.7)
            axes[i,0].set(title=family+' encoder stage',xlabel='Observed epoch',ylabel='Inner participant NLL')
            axes[i,1].set(title=family+' frozen-encoder head stage',xlabel='Observed epoch',ylabel='Inner participant NLL')
            axes[i,0].legend(ncol=2,fontsize=8)
        fig.suptitle(f'Selected candidates | seed {seed}'); fig.tight_layout(); plt.show()

## Scope and reporting
This experiment isolates the added residual scales against a **new single-order statistics-pooling control**. It does not prove superiority to original LPGNet without its separately run, matched evaluation. Report both arms, all seeds and all predictor/classifier parameter costs.

Keep the same curated cohort, outer folds and scoring unit when connecting to the separate LPGNet notebook. Published recording-level accuracy/F1 are not interchangeable with these participant metrics. Test transform/encode timings exclude input loading and precomputed downsampling: they are not complete end-to-end benchmarks.

The full protocol and departures from original LPGNet are in `LPR_EXTENSION_DESIGN.txt`. Export explicitly after analysis to preserve histories, fitted references, predictions and diagnostics. No new results are included in this unexecuted release.